# Alif Ensemble — capture FPS and transfer-time analysis

Parses TIFF filenames produced by [client.py](client.py) and reports:

- **Transfer duration** — `read_duration_ms` recorded per file (serial read time for the 921,776-byte payload).
- **Effective FPS** — derived from the delta between consecutive `read_start_timestamp_ms` values.

Filename schema: `{prefix}{index:04d}_{start_ts_ms}_{duration_ms}.tif`

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
DATA_DIR = Path("data_example")

# Trailing groups of the filename: ..._{index}_{start_ts_ms}_{duration_ms}.tif
PATTERN = re.compile(r"_(\d{4})_(\d+)_(\d+)\.tif$")

records = []
for path in sorted(DATA_DIR.iterdir()):
    m = PATTERN.search(path.name)
    if not m:
        continue
    idx, start_ms, dur_ms = m.groups()
    records.append(
        {
            "index": int(idx),
            "start_ts_ms": int(start_ms),
            "transfer_ms": int(dur_ms),
            "file": path.name,
        }
    )

df = pd.DataFrame.from_records(records).sort_values("index").reset_index(drop=True)
df["delta_ms"] = df["start_ts_ms"].diff()
df["fps"] = 1000.0 / df["delta_ms"]
print(f"Parsed {len(df)} files")
df.head()

## Summary statistics

In [ ]:
transfer = df["transfer_ms"]
fps = df["fps"].dropna()
delta = df["delta_ms"].dropna()

summary = pd.DataFrame(
    {
        "transfer_ms": [transfer.mean(), transfer.median(), transfer.std(), transfer.min(), transfer.max()],
        "frame_interval_ms": [delta.mean(), delta.median(), delta.std(), delta.min(), delta.max()],
        "fps": [fps.mean(), fps.median(), fps.std(), fps.min(), fps.max()],
    },
    index=["mean", "median", "std", "min", "max"],
).round(3)

wall_clock_s = (df["start_ts_ms"].iloc[-1] - df["start_ts_ms"].iloc[0]) / 1000.0
overall_fps = (len(df) - 1) / wall_clock_s if wall_clock_s else float("nan")

print(f"Frames: {len(df)}")
print(f"Wall-clock span: {wall_clock_s:.3f} s")
print(f"Overall FPS (frames-1 / span): {overall_fps:.3f}")
summary

## Visualizations

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(df["index"], df["transfer_ms"], color="tab:blue", linewidth=1)
axes[0].axhline(transfer.mean(), color="tab:red", linestyle="--", label=f"mean = {transfer.mean():.2f} ms")
axes[0].axhline(transfer.median(), color="tab:green", linestyle=":", label=f"median = {transfer.median():.2f} ms")
axes[0].set_ylabel("Transfer time (ms)")
axes[0].set_title("Per-frame serial transfer duration")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df["index"], df["fps"], color="tab:orange", linewidth=1)
axes[1].axhline(fps.mean(), color="tab:red", linestyle="--", label=f"mean = {fps.mean():.2f} fps")
axes[1].axhline(fps.median(), color="tab:green", linestyle=":", label=f"median = {fps.median():.2f} fps")
axes[1].set_xlabel("Frame index")
axes[1].set_ylabel("FPS")
axes[1].set_title("Instantaneous FPS (from consecutive start-timestamp deltas)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(transfer, bins=30, color="tab:blue", edgecolor="black", alpha=0.85)
axes[0].axvline(transfer.mean(), color="tab:red", linestyle="--", label=f"mean = {transfer.mean():.2f}")
axes[0].axvline(transfer.median(), color="tab:green", linestyle=":", label=f"median = {transfer.median():.2f}")
axes[0].set_xlabel("Transfer time (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Transfer-time distribution")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(fps, bins=30, color="tab:orange", edgecolor="black", alpha=0.85)
axes[1].axvline(fps.mean(), color="tab:red", linestyle="--", label=f"mean = {fps.mean():.2f}")
axes[1].axvline(fps.median(), color="tab:green", linestyle=":", label=f"median = {fps.median():.2f}")
axes[1].set_xlabel("FPS")
axes[1].set_title("FPS distribution")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["transfer_ms"], df["fps"], s=12, alpha=0.5)
plt.xlabel("Transfer time (ms)")
plt.ylabel("FPS")
plt.title("FPS vs. transfer time (per frame)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Throughput derived from transfer duration

Given the fixed payload of 921,776 bytes per frame (640×480×3 + 176-byte TIFF header).

In [ ]:
PAYLOAD_BYTES = 921_776
df["throughput_MBps"] = PAYLOAD_BYTES / (df["transfer_ms"] / 1000.0) / 1_000_000
print(
    f"Throughput  mean = {df['throughput_MBps'].mean():.3f} MB/s, "
    f"median = {df['throughput_MBps'].median():.3f} MB/s"
)
df[["index", "transfer_ms", "delta_ms", "fps", "throughput_MBps"]].describe().round(3)